In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import precision_recall_fscore_support
import warnings
from pytorch_dataset import EBSDStrainDataset

class MultiModalEBSDStrainCNN(nn.Module):
    """
    Improved ResNet architecture for EBSD patterns.
    """
    def __init__(self, euler_dim=3):
        super(MultiModalEBSDStrainCNN, self).__init__()

        # Load a pretrained ResNet18 for much faster and better convergence
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # ---------------------------------------------------------
        # CRITICAL FIX: EBSD patterns are likely small.
        # Standard ResNet uses a 7x7 conv (stride 2) + MaxPool (stride 2),
        # which downsamples the image by 4x right away. Total downsample is 32x.
        # This destroys fine diffraction features.
        # We replace the stem with a 3x3 conv (stride 1) and remove maxpool.
        # ---------------------------------------------------------
        resnet.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        resnet.maxpool = nn.Identity()

        # Extract features up to the adaptive average pooling
        self.image_features = nn.Sequential(*list(resnet.children())[:-1])

        self.euler_processor = nn.Sequential(
            nn.Linear(euler_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True)
        )

        # Classifier head (512 from resnet + 64 from euler)
        self.classifier = nn.Sequential(
            nn.Linear(512 + 64, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x, eulers):
        img_features = self.image_features(x)
        img_features = torch.flatten(img_features, 1)

        euler_features = self.euler_processor(eulers)

        combined = torch.cat((img_features, euler_features), dim=1)
        out = self.classifier(combined)
        return out.squeeze()

def calculate_metrics(y_true, y_pred, strain_classes=None):
    epsilon = 1e-8
    percent_errors = np.abs((y_true - y_pred) / (y_true + epsilon)) * 100
    avg_percent_error = np.mean(percent_errors)
    std_dev_error = np.std(percent_errors)

    if strain_classes is None:
        strain_classes = np.sort(np.unique(y_true))

    strain_to_int_map = {strain: i for i, strain in enumerate(strain_classes)}

    y_pred_snapped_floats = np.array([strain_classes[np.argmin(np.abs(strain_classes - pred))] for pred in y_pred])
    y_true_snapped_floats = np.array([strain_classes[np.argmin(np.abs(strain_classes - true))] for true in y_true])

    y_pred_classes_int = np.array([strain_to_int_map[s] for s in y_pred_snapped_floats])
    y_true_classes_int = np.array([strain_to_int_map[s] for s in y_true_snapped_floats])

    warnings.filterwarnings('ignore')
    precision, recall, f1, _ = precision_recall_fscore_support(y_true_classes_int, y_pred_classes_int, average='weighted')

    return avg_percent_error, std_dev_error, precision, recall, f1

def train_and_evaluate(h5_path="ebsd_fcc_fe.h5", epochs=100, batch_size=64, max_lr=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_dataset = EBSDStrainDataset(h5_path=h5_path, split="train")
    test_dataset = EBSDStrainDataset(h5_path=h5_path, split="test")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    model = MultiModalEBSDStrainCNN(euler_dim=3).to(device)

    # Combined Loss: MSE + Huber
    mse_loss = nn.MSELoss()
    huber_loss = nn.SmoothL1Loss()

    optimizer = optim.AdamW(model.parameters(), lr=max_lr, weight_decay=5e-4)

    # Cosine annealing scheduler usually outperforms OneCycleLR for finetuning ResNets
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=25, T_mult=2)

    best_f1 = 0.0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for batch_idx, (patterns, strains, eulers) in enumerate(train_loader):
            patterns = patterns.to(device)
            strains = strains.to(device)
            eulers = eulers.float().to(device)

            # Data Augmentation
            if torch.rand(1).item() > 0.5:
                patterns = torch.flip(patterns, dims=[-1]) # Horizontal flip
            if torch.rand(1).item() > 0.5:
                patterns = torch.flip(patterns, dims=[-2]) # Vertical flip

            noise = torch.randn_like(patterns) * 0.05
            patterns = patterns + noise

            optimizer.zero_grad()
            outputs = model(patterns, eulers)

            loss = 0.5 * mse_loss(outputs, strains) + 0.5 * huber_loss(outputs, strains)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Prevent exploding gradients
            optimizer.step()

        scheduler.step()

        # Evaluation
        model.eval()
        test_loss = 0.0
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for patterns, strains, eulers in test_loader:
                patterns = patterns.to(device)
                strains = strains.to(device)
                eulers = eulers.float().to(device)

                outputs = model(patterns, eulers)
                loss = 0.5 * mse_loss(outputs, strains) + 0.5 * huber_loss(outputs, strains)
                test_loss += loss.item()

                all_preds.extend(outputs.cpu().numpy())
                all_targets.extend(strains.cpu().numpy())

        test_loss /= len(test_loader)
        all_preds = np.array(all_preds)
        all_targets = np.array(all_targets)

        avg_pe, std_pe, precision, recall, f1 = calculate_metrics(all_targets, all_preds)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {running_loss/len(train_loader):.4f} - Test Loss: {test_loss:.4f}")
            print(f"Metrics: P: {precision:.4f} | R: {recall:.4f} | F1: {f1:.4f} | % Error: {avg_pe:.2f}% (Std: {std_pe:.2f})")

        if f1 > best_f1:
            best_f1 = f1
            save_path = "/content/drive/MyDrive/best_ebsd-strain_model.pth"
            torch.save(model.state_dict(), save_path)
            print("Saved to:", save_path)

    print(f"\nTraining completed. Best model saved. Best F1: {best_f1:.4f}")

if __name__ == "__main__":
    train_and_evaluate(h5_path="/content/drive/MyDrive/ebsd_fcc_fe.h5", epochs=100, batch_size=64)

Using device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 54.6MB/s]


Epoch [1/100] - Train Loss: 0.0000 - Test Loss: 35.2896
Metrics: P: 0.3040 | R: 0.3025 | F1: 0.2583 | % Error: 71.09% (Std: 59.88)
Saved to: /content/drive/MyDrive/best_ebsd-strain_model.pth
Saved to: /content/drive/MyDrive/best_ebsd-strain_model.pth
Saved to: /content/drive/MyDrive/best_ebsd-strain_model.pth
Epoch [5/100] - Train Loss: 0.0000 - Test Loss: 3.1062
Metrics: P: 0.7896 | R: 0.7633 | F1: 0.7452 | % Error: 27.21% (Std: 44.09)
Saved to: /content/drive/MyDrive/best_ebsd-strain_model.pth
Saved to: /content/drive/MyDrive/best_ebsd-strain_model.pth
Saved to: /content/drive/MyDrive/best_ebsd-strain_model.pth
Epoch [10/100] - Train Loss: 0.0000 - Test Loss: 3.2696
Metrics: P: 0.7727 | R: 0.7125 | F1: 0.6790 | % Error: 34.46% (Std: 52.05)
Saved to: /content/drive/MyDrive/best_ebsd-strain_model.pth
Epoch [15/100] - Train Loss: 0.0000 - Test Loss: 1.0329
Metrics: P: 0.9155 | R: 0.9067 | F1: 0.9039 | % Error: 14.84% (Std: 24.71)
Saved to: /content/drive/MyDrive/best_ebsd-strain_model.p